# Swahili Gemma 1B IT — Quickstart

This notebook shows how to load [Swahili Gemma 1B IT](https://www.kaggle.com/models/gracenoble/swahili-gemma-1b-it) and run English → Swahili translations.

The model is a fine-tuned [Gemma 3 1B](https://huggingface.co/google/gemma-3-1b-it), continued-pretrained and instruction-tuned for English→Swahili translation.

**Results on FLORES-200 devtest (English→Swahili, beam=8):**

| Model | BLEU | chrF++ |
|---|---|---|
| Gemma 3 1B base (no fine-tuning) | 0.41 | 14.28 |
| **This model** | **28.86** | **54.77** |

Also available as a quantized [Ollama release](https://ollama.com/amplifiedaccessorg/Swahili-gemma-1b-it) and on [Hugging Face](https://huggingface.co/AmplifiedAccess/Swahili-gemma-1b-it).

## 1. Setup

If you're running this as a Kaggle notebook with the model attached as an input (via "Add Input" → search for `swahili-gemma-1b-it`), you can skip the `kagglehub.model_download` call below and point directly at `/kaggle/input/swahili-gemma-1b-it/pytorch/default/1` instead.

In [ ]:
!pip install -q -U transformers kagglehub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import kagglehub

model_path = kagglehub.model_download("gracenoble/swahili-gemma-1b-it/pyTorch/default")
print("Model downloaded to:", model_path)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16 if device == "cuda" else torch.float32)
model.to(device)
model.eval()
print("Loaded on", device)

## 2. Translate

The model was trained on this exact prompt template for translation — using it gets the most reliable output. It also handles general Swahili instructions (from Aya/Alpaca-Swahili training data), but translation quality is best with this format.

In [ ]:
def translate(english_sentence, max_new_tokens=128):
    prompt = (
        "<start_of_turn>user\n"
        "Translate the following English sentence to Swahili.\n"
        f"English: {english_sentence.strip()}\nSwahili:<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=8,
            length_penalty=1.1,
            early_stopping=True,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


translate("The children are playing outside.")

## 3. A few more examples

In [ ]:
examples = [
    "I would like a cup of coffee, please.",
    "The government announced new policies to support small businesses across the country.",
    "Where is the nearest hospital?",
    "Farmers in this region grow maize, beans, and coffee.",
    "Thank you very much for your help today.",
]

for en in examples:
    sw = translate(en)
    print(f"EN: {en}\nSW: {sw}\n")

## 4. Batch translation

For translating many sentences, batch them together for better throughput. Left-padding is required for causal LM batch generation.

In [ ]:
def translate_batch(sentences, batch_size=8, max_new_tokens=128):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    results = []
    for i in range(0, len(sentences), batch_size):
        chunk = sentences[i:i + batch_size]
        prompts = [
            "<start_of_turn>user\n"
            "Translate the following English sentence to Swahili.\n"
            f"English: {s.strip()}\nSwahili:<end_of_turn>\n"
            "<start_of_turn>model\n"
            for s in chunk
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                num_beams=8, length_penalty=1.1, early_stopping=True,
            )
        decoded = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results += [d.strip() for d in decoded]
    return results


translate_batch(examples)

## Limitations

- Trained and evaluated only on **English→Swahili**; Swahili→English is untested and not claimed.
- Back-translation data augmentation was planned during training but did not execute in the training run that produced this checkpoint.
- Performance outside the training domains (news, general instructions) is unverified.
- As a 1B model, expect more errors on long, syntactically complex, or highly domain-specific sentences than from larger translation models.

See the [model page](https://www.kaggle.com/models/gracenoble/swahili-gemma-1b-it) for full training details and data sources.